In [1]:
# Importar librerías
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, year, month
import os

In [3]:
spark = SparkSession.builder \
    .appName("SECOP_Ingesta") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()
print(f"Spark Version: {spark.version}")
print(f"Spark Master: {spark.sparkContext.master}")

Spark Version: 3.5.0
Spark Master: spark://spark-master:7077


In [5]:
import requests
import json
from sodapy import Socrata

In [6]:
client = Socrata("www.datos.gov.co", "Xx8V8KQYOyztb44vVcE9efR3m")

In [7]:
results = client.get(
    "jbjy-vk9h",
    query="""
    SELECT *
    WHERE 
        fecha_de_firma > '2025-01-01T00:00:00'
    AND 
        fecha_de_firma < '2026-01-01T00:00:00'
    LIMIT 100000
    """
)

In [8]:
print("Registros descargados:", len(results))
print("Columnas:", results[0].keys())
# Guardar JSON localmente
json_path = "/opt/spark-data/raw/secop_contratos.json"
os.makedirs(os.path.dirname(json_path), exist_ok=True)

Registros descargados: 100000
Columnas: dict_keys(['nombre_entidad', 'nit_entidad', 'departamento', 'ciudad', 'localizaci_n', 'orden', 'sector', 'rama', 'entidad_centralizada', 'proceso_de_compra', 'id_contrato', 'referencia_del_contrato', 'estado_contrato', 'codigo_de_categoria_principal', 'descripcion_del_proceso', 'tipo_de_contrato', 'modalidad_de_contratacion', 'justificacion_modalidad_de', 'fecha_de_firma', 'fecha_de_inicio_del_contrato', 'fecha_de_fin_del_contrato', 'condiciones_de_entrega', 'tipodocproveedor', 'documento_proveedor', 'proveedor_adjudicado', 'es_grupo', 'es_pyme', 'habilita_pago_adelantado', 'liquidaci_n', 'obligaci_n_ambiental', 'obligaciones_postconsumo', 'reversion', 'origen_de_los_recursos', 'destino_gasto', 'valor_del_contrato', 'valor_de_pago_adelantado', 'valor_facturado', 'valor_pendiente_de_pago', 'valor_pagado', 'valor_amortizado', 'valor_pendiente_de', 'valor_pendiente_de_ejecucion', 'estado_bpin', 'c_digo_bpin', 'anno_bpin', 'saldo_cdp', 'saldo_vigenci

In [9]:
with open(json_path, 'w', encoding='utf-8') as f:
    for record in results:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

In [10]:
print(f"Datos guardados en: {json_path}")

Datos guardados en: /opt/spark-data/raw/secop_contratos.json


In [11]:
print("Leyendo datos desde JSON...")
df_raw = spark.read.json(json_path)

print(f"Total de registros: {df_raw.count()}")
print(f"Total de columnas: {len(df_raw.columns)}")

Leyendo datos desde JSON...


Total de registros: 100000
Total de columnas: 87


In [12]:
# Explorar esquema del dataset
print("\n=== ESQUEMA DEL DATASET ===")
df_raw.printSchema()
import unicodedata
import re


=== ESQUEMA DEL DATASET ===
root
 |-- anno_bpin: string (nullable = true)
 |-- c_digo_bpin: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- codigo_de_categoria_principal: string (nullable = true)
 |-- codigo_entidad: string (nullable = true)
 |-- codigo_proveedor: string (nullable = true)
 |-- condiciones_de_entrega: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- descripcion_del_proceso: string (nullable = true)
 |-- descripcion_documentos_tipo: string (nullable = true)
 |-- destino_gasto: string (nullable = true)
 |-- dias_adicionados: string (nullable = true)
 |-- documento_proveedor: string (nullable = true)
 |-- documentos_tipo: string (nullable = true)
 |-- domicilio_representante_legal: string (nullable = true)
 |-- duraci_n_del_contrato: string (nullable = true)
 |-- el_contrato_puede_ser_prorrogado: string (nullable = true)
 |-- entidad_centralizada: string (nullable = true)
 |-- es_grupo: string (nullable = true)
 |-- es_pyme: st

In [13]:
def normalize_col(c):
    c = unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode("ascii")
    c = re.sub(r"[^a-zA-Z0-9_]", "_", c)
    return c.lower()


In [14]:
df_silver = df_raw.toDF(*[normalize_col(c) for c in df_raw.columns])
from pyspark.sql.functions import col, to_timestamp, regexp_replace
from functools import reduce

In [15]:
df_silver = reduce(lambda d, c:
    d.withColumn(c,
        to_timestamp(col(c)) if "fecha" in c
        else col(c).cast("double") if any(x in c for x in ["valor", "saldo"])
        else col(c).cast("int") if any(x in c for x in ["dias", "duracion", "plazo"])
        else col(c)
    ),
    df_silver.columns,
    df_silver
)
if "urlproceso" in df_silver.columns:
    df_silver = df_silver.withColumn("url_proceso", col("urlproceso.url")) \
           .drop("urlproceso")
from pyspark.sql.functions import col
df_silver = df_silver \
    .withColumn("duraci_n_del_contrato", col("duraci_n_del_contrato").cast("int")) \
    .withColumn("recursos_de_credito", col("recursos_de_credito").cast("double")) \
    .withColumn("recursos_propios", col("recursos_propios").cast("double")) \
    .withColumn("presupuesto_general_de_la_nacion_pgn", col("presupuesto_general_de_la_nacion_pgn").cast("double")) \
    .withColumn("sistema_general_de_participaciones", col("sistema_general_de_participaciones").cast("double")) \
    .withColumn("sistema_general_de_regal_as", col("sistema_general_de_regal_as").cast("double"))

In [16]:
print("=== ESQUEMA SILVER ===")
df_silver.printSchema()
# Registros inválidos críticos
df_invalid = df_silver.filter(
    col("valor_del_contrato").isNull() |
    col("fecha_de_firma").isNull() |
    col("departamento").isNull()
)

=== ESQUEMA SILVER ===
root
 |-- anno_bpin: string (nullable = true)
 |-- c_digo_bpin: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- codigo_de_categoria_principal: string (nullable = true)
 |-- codigo_entidad: string (nullable = true)
 |-- codigo_proveedor: string (nullable = true)
 |-- condiciones_de_entrega: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- descripcion_del_proceso: string (nullable = true)
 |-- descripcion_documentos_tipo: string (nullable = true)
 |-- destino_gasto: string (nullable = true)
 |-- dias_adicionados: integer (nullable = true)
 |-- documento_proveedor: string (nullable = true)
 |-- documentos_tipo: string (nullable = true)
 |-- domicilio_representante_legal: string (nullable = true)
 |-- duraci_n_del_contrato: integer (nullable = true)
 |-- el_contrato_puede_ser_prorrogado: string (nullable = true)
 |-- entidad_centralizada: string (nullable = true)
 |-- es_grupo: string (nullable = true)
 |-- es_pyme: string

In [17]:
print("Registros inválidos:", df_invalid.count())

Registros inválidos: 0


In [18]:
# Porcentaje de calidad
total = df_silver.count()
invalid = df_invalid.count()
print(f"Calidad del dataset: {(1 - invalid/total)*100:.2f}%")

Calidad del dataset: 100.00%


In [19]:
# Mostrar primeras filas
print("\n=== PRIMERAS 5 FILAS ===")
df_silver.show(5, truncate=False)
# Mostrar nombres de columnas
print("\n=== COLUMNAS DISPONIBLES ===")
for col_name in df_silver.columns:
    print(f"- {col_name}")



=== PRIMERAS 5 FILAS ===


26/02/12 23:59:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+-----------+-----------+-----------------------------+--------------+----------------+----------------------+--------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+--------------+----------------+-------------------+---------------+-----------------------------+---------------------+--------------------------------+--------------------+--------+-------+---------------+-----------+---------------+-------------------------+-------------------+----------------------------+-------------------------------------+---------------------+------------------------+--------------------------+------------------------+------------------+----------------------------------+--------------------------+-----------+-----------------------------+-----------------------------+-------

In [20]:
print("\n=== INFORMACIÓN DEL DATASET ===")
print(f"Registros totales: {df_silver.count():,}")
print(f"Columnas totales: {len(df_silver.columns)}")


=== INFORMACIÓN DEL DATASET ===


Registros totales: 100,000
Columnas totales: 87


In [21]:
columnas_clave = [
    "referencia_del_contrato",
    "nit_entidad",
    "nombre_entidad",
    "departamento",
    "ciudad",
    "tipo_de_contrato",
    "valor_del_contrato",
    "fecha_de_firma",
    "duraci_n_del_contrato",  
    "proveedor_adjudicado",     
    "estado_contrato"
]
columnas_disponibles = [col for col in columnas_clave if col in df_silver.columns]
print(f"\n=== COLUMNAS SELECCIONADAS ({len(columnas_disponibles)}) ===")
for col in columnas_disponibles:
    print(f"- {col}")
# Filtrar columnas disponibles
if columnas_disponibles:
    df_clean = df_silver.select(*columnas_disponibles)
else:
    # Si no encontramos las columnas esperadas, usamos todas
    print("ADVERTENCIA: No se encontraron las columnas esperadas. Usando todas las columnas.")
    df_clean = df_silver


=== COLUMNAS SELECCIONADAS (11) ===
- referencia_del_contrato
- nit_entidad
- nombre_entidad
- departamento
- ciudad
- tipo_de_contrato
- valor_del_contrato
- fecha_de_firma
- duraci_n_del_contrato
- proveedor_adjudicado
- estado_contrato


In [22]:
# ----------------------------
# 10. Particionar por fecha
# ----------------------------
df_clean = df_clean \
    .withColumn("year", year("fecha_de_firma")) \
    .withColumn("month", month("fecha_de_firma"))
output_path = "/opt/spark-data/processed/secop_contratos_silver.parquet"
print(f"\n=== GUARDANDO EN FORMATO PARQUET ===")
print(f"Ruta: {output_path}")


=== GUARDANDO EN FORMATO PARQUET ===
Ruta: /opt/spark-data/processed/secop_contratos_silver.parquet


In [23]:
df_clean.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet(output_path)

print("Datos guardados exitosamente en formato Parquet")


Datos guardados exitosamente en formato Parquet


In [24]:
print("\n=== VERIFICACIÓN ===")
df_verificacion = spark.read.parquet(output_path)
print(f"Registros en Parquet: {df_verificacion.count():,}")
print(f"Columnas en Parquet: {len(df_verificacion.columns)}")


=== VERIFICACIÓN ===
Registros en Parquet: 100,000
Columnas en Parquet: 13


In [25]:
df_check = spark.read.parquet(output_path)

In [26]:
print("\n=== VALIDACIÓN POST-INGESTA ===")
df_check.printSchema()
print("\n=== MUESTRA FINAL ===")
df_check.show(5, truncate=False)
print("\n=== RESUMEN FINAL ===")
print(f"Registros: {df_check.count():,}")
print(f"Columnas: {len(df_check.columns)}")
print("Particiones:", df_check.select("year","month").distinct().count())
# Resumen final
print("\n" + "="*60)
print("RESUMEN DE INGESTA")
print("="*60)
print(f"✓ Datos descargados desde API Socrata")
print(f"✓ Registros procesados: {df_clean.count():,}")
print(f"✓ Formato de salida: Parquet")
print(f"✓ Ubicación: {output_path}")
print("="*60)


=== VALIDACIÓN POST-INGESTA ===
root
 |-- referencia_del_contrato: string (nullable = true)
 |-- nit_entidad: string (nullable = true)
 |-- nombre_entidad: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- tipo_de_contrato: string (nullable = true)
 |-- valor_del_contrato: double (nullable = true)
 |-- fecha_de_firma: timestamp (nullable = true)
 |-- duraci_n_del_contrato: integer (nullable = true)
 |-- proveedor_adjudicado: string (nullable = true)
 |-- estado_contrato: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)


=== MUESTRA FINAL ===
+-----------------------+-----------+--------------------------------------------------+--------------------------+-----------+-----------------------+------------------+-------------------+---------------------+------------------------------+---------------+----+-----+
|referencia_del_contrato|nit_entidad|nombre_entidad               

Particiones: 1

RESUMEN DE INGESTA
✓ Datos descargados desde API Socrata


✓ Registros procesados: 100,000
✓ Formato de salida: Parquet
✓ Ubicación: /opt/spark-data/processed/secop_contratos_silver.parquet


In [27]:
# Detener SparkSession
spark.stop()
print("SparkSession finalizada")

SparkSession finalizada
